<a href="https://colab.research.google.com/github/PedroHS05/am-t4-s1a2026/blob/main/Corre%C3%A7%C3%A3o_da_base_de_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import pickle
import pandas as pd

In [14]:
base_de_dados = pd.read_excel('/content/drive/MyDrive/atividade_data_mining/base_de_dados.xlsx')

In [16]:
import re
import numpy as np
import pandas as pd

# 1. Carregar a base de dados do Google Drive
caminho_entrada = (
    '/content/drive/MyDrive/atividade_data_mining/base_de_dados.xlsx'
)
df = pd.read_excel(caminho_entrada)

print('--- Resumo Inicial ---')
print(f'Total de registros carregados: {len(df)}')

df_clean = df.copy()

# ==========================================
# 0. TRATAMENTO E HIGIENIZAÇÃO DOS NOMES (NOVO)
# ==========================================
def limpar_nome(nome):
  if pd.isna(nome):
    return nome
  # Remove tags HTML (ex: <b>João Alves</b> -> João Alves) e espaços extras
  nome_limpo = re.sub(r'<[^>]*>', '', str(nome)).strip()

  # Converte para Title Case mantendo preposições em minúsculo
  palavras = nome_limpo.split()
  palavras_formatadas = []
  for p in palavras:
    if p.lower() in ['de', 'da', 'do', 'das', 'dos']:
      palavras_formatadas.append(p.lower())
    else:
      palavras_formatadas.append(p.capitalize())
  return ' '.join(palavras_formatadas)


df_clean['nome_completo'] = df_clean['nome_completo'].apply(limpar_nome)

# ==========================================
# 1. PADRONIZAÇÃO DE VARIÁVEIS CATEGÓRICAS
# ==========================================

# Gênero
mapa_genero = {
    'Mulher': 'Feminino',
    'Feminino': 'Feminino',
    'feminino': 'Feminino',
    'FEMININO': 'Feminino',
    'F': 'Feminino',
    'Fem': 'Feminino',
    'MASCULINO': 'Masculino',
    'Masculino': 'Masculino',
    'masculino': 'Masculino',
    'Homem': 'Masculino',
    'M': 'Masculino',
    'Masc': 'Masculino',
    'MASCULIN0': 'Masculino',
}
df_clean['genero'] = (
    df_clean['genero']
    .astype(str)
    .str.strip()
    .map(mapa_genero)
    .fillna('Não Informado')
)

# Estado (UF)
df_clean['estado'] = df_clean['estado'].astype(str).str.strip().str.upper()
df_clean['estado'] = df_clean['estado'].replace({
    'RIO DE JANEIRO': 'RJ',
    'NONE': np.nan,
    'N/A': np.nan,
    'NULL': np.nan,
})

# Cidade (Corrigindo nomes truncados antes de preencher ausentes)
cidade_map = {
    'Belo Horizor': 'Belo Horizonte',
    'Belo Horizor MG': 'Belo Horizonte',
    'Rio de Janei': 'Rio de Janeiro',
    'None': np.nan,
    'N/A': np.nan,
    'NULL': np.nan,
}
df_clean['cidade'] = (
    df_clean['cidade'].replace(cidade_map).fillna('Não Informado')
)

# ==========================================
# 2. CONVERSÃO DE TIPOS E TRATAMENTO DE OUTLIERS
# ==========================================

df_clean['data_nascimento'] = pd.to_datetime(
    df_clean['data_nascimento'], errors='coerce'
)
df_clean['data_cadastro'] = pd.to_datetime(
    df_clean['data_cadastro'], errors='coerce'
)

# Idade (Remover < 0 ou > 120 e recalcular se necessário)
df_clean['idade'] = pd.to_numeric(df_clean['idade'], errors='coerce')
df_clean.loc[(df_clean['idade'] < 0) | (df_clean['idade'] > 120), 'idade'] = (
    np.nan
)
idade_calculada = 2026 - df_clean['data_nascimento'].dt.year
df_clean['idade'] = df_clean['idade'].fillna(idade_calculada)


# Salário Mensal (Trata símbolos R$, espaços, textos 'un'/'?' e vírgulas decimais)
def limpar_salario(val):
  if pd.isna(val):
    return np.nan
  val_str = str(val).replace('R$', '').replace(' ', '').strip()
  if ',' in val_str and '.' in val_str:
    val_str = val_str.replace('.', '').replace(',', '.')
  elif ',' in val_str:
    val_str = val_str.replace(',', '.')
  try:
    v = float(val_str)
    if 0 < v <= 1000000:
      return v
    return np.nan
  except:
    return np.nan


df_clean['salario_mensal'] = df_clean['salario_mensal'].apply(limpar_salario)


# Total Compras (Trata sufixo 'un' e converte em número)
def limpar_compras(val):
  if pd.isna(val):
    return np.nan
  val_str = str(val).lower().replace('un', '').strip()
  try:
    return float(val_str)
  except:
    return np.nan


df_clean['total_compras'] = df_clean['total_compras'].apply(limpar_compras)

# Score de Crédito (Manter apenas faixa válida de 0 a 1000)
df_clean['score_credito'] = pd.to_numeric(
    df_clean['score_credito'], errors='coerce'
)
df_clean.loc[
    (df_clean['score_credito'] < 0) | (df_clean['score_credito'] > 1000),
    'score_credito',
] = np.nan

# ==========================================
# 3. IMPUTAÇÃO DE VALORES AUSENTES (MEDIANA)
# ==========================================

for col in ['idade', 'salario_mensal', 'score_credito', 'total_compras']:
  df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Formatação final
df_clean['idade'] = df_clean['idade'].round().astype(int)
df_clean['salario_mensal'] = df_clean['salario_mensal'].round(2)
df_clean['score_credito'] = df_clean['score_credito'].round(1)
df_clean['total_compras'] = df_clean['total_compras'].round().astype(int)

df_clean['data_nascimento'] = df_clean['data_nascimento'].dt.strftime(
    '%Y-%m-%d'
)
df_clean['data_cadastro'] = df_clean['data_cadastro'].dt.strftime('%Y-%m-%d')

# ==========================================
# 4. EXPORTAÇÃO DOS ARQUIVOS (TXT E EXCEL)
# ==========================================

pasta_destino = '/content/drive/MyDrive/atividade_data_mining/'

# Exportar TXT (Separado por ponto e vírgula)
caminho_txt = pasta_destino + 'base_de_dados_higienizada.txt'
df_clean.to_csv(caminho_txt, sep=';', index=False, encoding='utf-8')

# Exportar Excel (.xlsx)
caminho_excel = pasta_destino + 'base_de_dados_higienizada.xlsx'
df_clean.to_excel(caminho_excel, index=False, sheet_name='base_higienizada')

print(' Processo concluído com sucesso!')
print(f'📄 Arquivo TXT salvo em: {caminho_txt}')
print(f'📊 Arquivo Excel salvo em: {caminho_excel}')

--- Resumo Inicial ---
Total de registros carregados: 5000
 Processo concluído com sucesso!
📄 Arquivo TXT salvo em: /content/drive/MyDrive/atividade_data_mining/base_de_dados_higienizada.txt
📊 Arquivo Excel salvo em: /content/drive/MyDrive/atividade_data_mining/base_de_dados_higienizada.xlsx
